# Stationary vs. tap feature exploration

This exploratory notebook compares one-second RMS and peak-to-peak windows for stationary and table-tap recordings. Run it from this notebook's directory; the data paths below point back to the repository-level `data/` folder.

> **Pipeline note:** The maintained feature implementation is in `src/machine_sentinel/features.py`. This notebook is retained for interactive exploration and historical context, so production changes should be made in the shared module first.

In [ ]:
import pandas as pd
import numpy as np

SAMPLE_RATE = 200
WINDOW_SIZE = 200       # 1 second
COUNTS_PER_G = 4096.0   # MPU6050 at ±8g


def extract_features(df, experiment, label=None):

    features = []

    for start in range(0, len(df), WINDOW_SIZE):

        end = start + WINDOW_SIZE

        # Ignore incomplete final window
        if end > len(df):
            break

        window = df.iloc[start:end]

        # Convert raw Z-axis counts -> g
        z_g = window["az"] / COUNTS_PER_G

        # Remove gravity / DC component
        z_centered = z_g - z_g.mean()

        # Time-domain features
        rms = np.sqrt(np.mean(np.square(z_centered)))

        # ddof=0 so STD matches centered RMS mathematically
        std = z_centered.std(ddof=0)

        peak_to_peak = z_g.max() - z_g.min()

        features.append({
            "start_idx": start,
            "end_idx": end,
            "start_time_s": start / SAMPLE_RATE,

            "rms": rms,
            "std": std,
            "ptp": peak_to_peak,

            "experiment": experiment,
            "label": label
        })

    return pd.DataFrame(features)

In [ ]:
df_stationary = pd.read_csv("../../data/raw/stationary.csv")

stationary_features = extract_features(
    df_stationary,
    experiment="stationary",
    label="normal"
)

print(stationary_features)

In [ ]:
df_tap = pd.read_csv("../../data/raw/table_taps.csv")

tap_features = extract_features(
    df_tap,
    experiment="tap_test",
    label=None
)

print(tap_features)

In [ ]:
all_features = pd.concat(
    [stationary_features, tap_features],
    ignore_index=True
)

print(all_features.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

plt.plot(
    stationary_features["start_time_s"],
    stationary_features["rms"],
    marker="o"
)

plt.xlabel("Time (s)")
plt.ylabel("Centered RMS (g)")
plt.title("Stationary - RMS per 1-second window")

plt.show()

In [ ]:
plt.figure()

plt.plot(
    tap_features["start_time_s"],
    tap_features["rms"],
    marker="o"
)

plt.xlabel("Time (s)")
plt.ylabel("Centered RMS (g)")
plt.title("Tap Test - RMS per 1-second window")

plt.show()

In [ ]:
plt.figure()

plt.scatter(
    stationary_features["rms"],
    stationary_features["ptp"],
    label="Stationary"
)

plt.scatter(
    tap_features["rms"],
    tap_features["ptp"],
    label="Tap experiment"
)

plt.xlabel("Centered RMS (g)")
plt.ylabel("Peak-to-Peak (g)")
plt.title("Vibration Feature Space")

plt.legend()
plt.show()